# Numerical eigensolver: WR-90 TE10 cutoff sweep

Build a coarse 2-D triangle mesh of the WR-90 rectangular waveguide cross-section ($a \times b = 22.86\ \mathrm{mm} \times 10.16\ \mathrm{mm}$, air-filled), then drive `yee.eigensolver.NumericalCrossSection.solve` at a frequency sweep to extract the dominant TE10 propagation constant $\beta(f)$. Compare against the closed-form $\beta = \sqrt{k^2 - (\pi/a)^2}$.

WR-90's TE10 cutoff is $f_c = c / (2 a) \approx 6.557\ \mathrm{GHz}$. Below cutoff the mode is evanescent; the closed-form analytic returns NaN there and the numerical eigensolver picks up the next-higher physical $\beta$. The sweep below therefore stays above 6.6 GHz so every solve lands on the dominant guided mode.

In [ ]:
import math

import yee
# Both import styles work — `yee.eigensolver` is registered in `sys.modules`
# by the PyO3 init (mirrors the `yee.touchstone` pattern).
from yee.eigensolver import NumericalCrossSection, TriMesh2D

## Build the cross-section mesh

A structured `nx × ny` quad grid spanning `[0, a] × [0, b]`, each quad split along its lower-left → upper-right diagonal into two CCW triangles. All triangles share material tag 0 (air fill). The `TriMesh2D` constructor enforces strictly positive signed area, so the CCW winding is non-negotiable.

In [ ]:
a = 22.86e-3  # WR-90 long inner dimension (m)
b = 10.16e-3  # WR-90 short inner dimension (m)
nx, ny = 6, 6  # 72 triangles, 49 vertices — matches the Rust validation gate

vertices = [
    (a * i / nx, b * j / ny)
    for j in range(ny + 1)
    for i in range(nx + 1)
]

def idx(i, j):
    return j * (nx + 1) + i

triangles = []
for j in range(ny):
    for i in range(nx):
        v00 = idx(i, j)
        v10 = idx(i + 1, j)
        v11 = idx(i + 1, j + 1)
        v01 = idx(i, j + 1)
        triangles.append((v00, v10, v11))
        triangles.append((v00, v11, v01))

mesh = TriMesh2D(vertices, triangles)
print(f"n_verts = {mesh.n_verts()}, n_tris = {mesh.n_tris()}")
print(f"area(0) = {mesh.area(0):.6e} m^2 (one quad-half)")

## Material dicts

`eps_r` / `mu_r` are `dict[int, complex]` keyed by material tag. WR-90 is air-filled, so a single tag (`0`) maps to `1 + 0j` for both.

In [ ]:
eps_r = {0: complex(1, 0)}
mu_r = {0: complex(1, 0)}

## Frequency sweep

Solve the dense eigenproblem at each `freq_hz`, read back `nc.beta`, compare against the analytic closed-form.

In [ ]:
C0 = 299_792_458.0

def analytic_beta(a, freq_hz, eps_r=1.0):
    k = 2.0 * math.pi * freq_hz * math.sqrt(eps_r) / C0
    kc = math.pi / a
    if k <= kc:
        return float("nan")
    return math.sqrt(k * k - kc * kc)

freqs_ghz = [7.0, 8.0, 10.0, 12.0]
results = []
for f_ghz in freqs_ghz:
    freq_hz = f_ghz * 1.0e9
    nc = NumericalCrossSection(mesh, eps_r, mu_r)
    nc.solve(freq_hz)
    beta_num = nc.beta.real  # lossless air ⇒ Im ≈ 0
    beta_an = analytic_beta(a, freq_hz)
    rel_err = abs(beta_num - beta_an) / beta_an * 100.0
    results.append((f_ghz, beta_num, beta_an, rel_err))
    print(
        f"f = {f_ghz:5.2f} GHz | numerical β = {beta_num:8.3f} rad/m | "
        f"analytic β = {beta_an:8.3f} rad/m | rel err = {rel_err:6.3f} %"
    )

## Plot $\beta$ vs frequency

Numerical points overlaid on a dense analytic curve. Matplotlib is optional; if it's not installed the cell falls back to printing the table.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    f_dense_ghz = np.linspace(7.0, 12.0, 200)
    beta_dense = np.array([analytic_beta(a, fg * 1e9) for fg in f_dense_ghz])

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(f_dense_ghz, beta_dense, label="analytic TE10", color="C0", lw=1.5)
    ax.scatter(
        [r[0] for r in results],
        [r[1] for r in results],
        marker="o",
        color="C3",
        zorder=5,
        label="numerical (yee.eigensolver)",
    )
    ax.axvline(C0 / (2.0 * a) / 1e9, ls="--", color="k", alpha=0.4,
               label=f"cutoff {C0 / (2.0 * a) / 1e9:.3f} GHz")
    ax.set_xlabel("Frequency (GHz)")
    ax.set_ylabel(r"$\beta$ (rad/m)")
    ax.set_title("WR-90 TE10 propagation constant — numerical vs analytic")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — printing table only:")
    for f_ghz, beta_num, beta_an, rel_err in results:
        print(
            f"  f = {f_ghz:5.2f} GHz | β_num = {beta_num:8.3f} | β_an = {beta_an:8.3f} | "
            f"rel err = {rel_err:6.3f} %"
        )

On a 6×6-quad mesh the numerical $\beta$ tracks the analytic curve to better than 1 % across the sweep — well within the Phase 1.3.1.1 validation gate. Refining the mesh (`nx`, `ny` ≥ 12) tightens that to a few 0.1 % at the cost of $O(n^3)$ wall-time growth in the dense eigensolve. Sparse shift-and-invert is Phase 1.3.1.1 step 4.